# 04. Full lesson: lokalny LLM, RAG, ChromaDB, workflow i trening w jednym notebooku

To jest wersja demonstracyjna „wszystko w jednym” na zajęcia.

**Kolejność:**

1. instalacja,
2. lokalny LLM w Colabie,
3. mini RAG TF-IDF,
4. RAG z ChromaDB,
5. router workflow,
6. mała sieć neuronowa do klasyfikacji ścieżki.

In [1]:
!pip -q install transformers accelerate sentencepiece sentence-transformers chromadb scikit-learn pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the so

In [2]:
import os
import shutil
import torch
import pandas as pd
import numpy as np

print("GPU dostępne:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU dostępne: True
GPU: Tesla T4


# Część A. Lokalny LLM w Colabie

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Urządzenie:", device)

# Uwaga: na GPU używamy float16, na CPU float32.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)
model.to(device)
model.eval()

print("Model załadowany:", MODEL_NAME)

Urządzenie: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model załadowany: Qwen/Qwen2.5-0.5B-Instruct


In [4]:
def ask_llm(prompt, system="Jesteś pomocnym asystentem dydaktycznym. Odpowiadasz po polsku, krótko i precyzyjnie.", max_new_tokens=250):
    """Prosta funkcja do rozmowy z lokalnym modelem uruchomionym w Colabie."""
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True)
    return answer.strip()

In [5]:
print(ask_llm("W 5 punktach wyjaśnij, czym jest lokalny LLM."))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Localmente (localmente) to znaczy, że w tym miejscu lub lokalizacji. Jest to termin, który może być używany w różnych kontekstach, ale nie jest zawsze jednym z najważniejszych konkretnych terminów. W zależności od kontekstu, lokalność może być używana do:

1. Zdefiniowania lokalnej sytuacji lub situacji.
2. Definicji lokalnego stanu lub zachodu.
3. Zastosowania lokalnych metod lub technologii.

Jeśli chcesz oto przykład, jak to używać w języku angielskim:

"Localmente, " to znaczy "w tej lokalizacji". 

Oto kilka przykładów:

- "Localmente, ilustra l'importanza della sicurezza delle informazioni personali." - "Localmente, przedstawiam, jak ważne è la sicurezza delle informazioni personali."

- "Localmente, il sistema ha ottenuto successo." - "Localmente, l'azienda ha raggiunto successo."

- "Localmente, il lavoro è stato completato in tempo." -


# Część B. Dokumenty dla RAG

In [6]:
documents = [
    "Student może mieć maksymalnie dwie nieobecności.",
    "Trzecia nieobecność wymaga wykonania zadania dodatkowego.",
    "Projekt końcowy musi zostać oddany do 30 czerwca.",
    "Projekt może być wykonany indywidualnie albo w parach.",
    "Użycie ChatGPT jest dozwolone, ale student musi rozumieć kod.",
    "Plagiat powoduje brak zaliczenia.",
    "Aktywność na zajęciach może podnieść ocenę końcową o pół stopnia."
]

for i, d in enumerate(documents):
    print(i, d)

0 Student może mieć maksymalnie dwie nieobecności.
1 Trzecia nieobecność wymaga wykonania zadania dodatkowego.
2 Projekt końcowy musi zostać oddany do 30 czerwca.
3 Projekt może być wykonany indywidualnie albo w parach.
4 Użycie ChatGPT jest dozwolone, ale student musi rozumieć kod.
5 Plagiat powoduje brak zaliczenia.
6 Aktywność na zajęciach może podnieść ocenę końcową o pół stopnia.


# Część C. Mini RAG TF-IDF

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_tfidf(query, documents, n_results=2):
    vectorizer = TfidfVectorizer()
    doc_vectors = vectorizer.fit_transform(documents)
    query_vector = vectorizer.transform([query])
    similarities = cosine_similarity(query_vector, doc_vectors).flatten()
    best_indices = similarities.argsort()[::-1][:n_results]
    return [(documents[i], float(similarities[i])) for i in best_indices]

def rag_tfidf(query, documents, n_results=2):
    selected = retrieve_tfidf(query, documents, n_results=n_results)
    selected_docs = [doc for doc, score in selected]
    context = " ".join(selected_docs)

    prompt = f"""Odpowiedz na pytanie wyłącznie na podstawie kontekstu.

KONTEKST:
{context}

PYTANIE:
{query}

Jeżeli w kontekście nie ma odpowiedzi, napisz:
"Nie wiem na podstawie dostarczonych dokumentów."
"""
    answer = ask_llm(prompt, max_new_tokens=180)
    return answer, selected

query = "Czy mogę użyć ChatGPT w projekcie?"
answer, selected = rag_tfidf(query, documents)

print("FRAGMENTY:")
for doc, score in selected:
    print(f"{score:.3f} | {doc}")

print("ODPOWIEDŹ:")
print(answer)

FRAGMENTY:
0.345 | Użycie ChatGPT jest dozwolone, ale student musi rozumieć kod.
0.000 | Aktywność na zajęciach może podnieść ocenę końcową o pół stopnia.
ODPOWIEDŹ:
Nie wiem na podstawie dostarczonych dokumentów.


# Część D. RAG z ChromaDB

In [9]:
from sentence_transformers import SentenceTransformer
import chromadb

embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
embeddings = embedding_model.encode(documents).tolist()

DB_PATH = "/content/rag_db_full"
if os.path.exists(DB_PATH):
    shutil.rmtree(DB_PATH)

chroma_client = chromadb.PersistentClient(path=DB_PATH)
collection = chroma_client.get_or_create_collection("regulamin")

collection.add(
    ids=[f"doc_{i}" for i in range(len(documents))],
    documents=documents,
    embeddings=embeddings
)

print("ChromaDB gotowa.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ChromaDB gotowa.


In [10]:
def retrieve_chroma(query, collection, embedding_model, n_results=3):
    query_embedding = embedding_model.encode([query]).tolist()[0]
    results = collection.query(query_embeddings=[query_embedding], n_results=n_results)
    return results["documents"][0]

def rag_chroma(query, collection, embedding_model, n_results=3):
    retrieved_docs = retrieve_chroma(query, collection, embedding_model, n_results=n_results)
    context = " ".join(retrieved_docs)

    prompt = f"""
Odpowiedz na pytanie wyłącznie na podstawie kontekstu.

KONTEKST:
{context}

PYTANIE:
{query}

ZASADY:
- Nie dopowiadaj niczego spoza kontekstu.
- Jeżeli odpowiedzi nie ma w kontekście, napisz: "Nie wiem na podstawie dostarczonych dokumentów."
- Odpowiedz po polsku.
"""
    answer = ask_llm(prompt, max_new_tokens=200)
    return answer, retrieved_docs

for q in [
    "Czy student z trzema nieobecnościami może zaliczyć kurs?",
    "Czy projekt można robić w parach?",
    "Czy egzamin jest ustny?"
]:
    print("=" * 90)
    print("PYTANIE:", q)
    answer, docs = rag_chroma(q, collection, embedding_model)
    print("FRAGMENTY:")
    for d in docs:
        print("-", d)
    print("ODPOWIEDŹ:")
    print(answer)

PYTANIE: Czy student z trzema nieobecnościami może zaliczyć kurs?
FRAGMENTY:
- Student może mieć maksymalnie dwie nieobecności.
- Trzecia nieobecność wymaga wykonania zadania dodatkowego.
- Aktywność na zajęciach może podnieść ocenę końcową o pół stopnia.
ODPOWIEDŹ:
Tak, student może zaliczyć kurs, jeśli jego ocena końcowa jest podzielona o pół stopnia.
PYTANIE: Czy projekt można robić w parach?
FRAGMENTY:
- Projekt może być wykonany indywidualnie albo w parach.
- Projekt końcowy musi zostać oddany do 30 czerwca.
- Plagiat powoduje brak zaliczenia.
ODPOWIEDŹ:
Tak, projekt może być wykonany w parach.
PYTANIE: Czy egzamin jest ustny?
FRAGMENTY:
- Aktywność na zajęciach może podnieść ocenę końcową o pół stopnia.
- Plagiat powoduje brak zaliczenia.
- Użycie ChatGPT jest dozwolone, ale student musi rozumieć kod.
ODPOWIEDŹ:
Ja nie widzę tego dokumentu.


# Część E. Router workflow

In [11]:
def workflow_router(question):
    q = question.lower()
    document_words = ["dokument", "pdf", "regulamin", "instrukcja", "procedura", "sylabus", "umowa"]
    calculation_words = ["policz", "oblicz", "średnia", "suma", "wykres", "csv", "excel", "tabela"]
    training_words = ["trening", "trenować", "dostroić", "fine-tuning", "nauczyć model", "mój styl"]

    if any(word in q for word in document_words):
        return "RAG", "Pytanie dotyczy dokumentów lub bazy wiedzy."
    elif any(word in q for word in calculation_words):
        return "TOOL", "Pytanie wymaga obliczeń albo pracy na danych."
    elif any(word in q for word in training_words):
        return "TRAINING", "Pytanie dotyczy uczenia lub dostrajania modelu."
    else:
        return "PROMPT", "Wystarczy zwykła odpowiedź modelu."

for case in [
    "Wyjaśnij mi, czym jest LLM.",
    "Mam dokument PDF i chcę pytać o jego treść.",
    "Policz średnią ocen z pliku CSV.",
    "Chcę dostroić model do mojego stylu pisania."
]:
    category, reason = workflow_router(case)
    print(case, "=>", category, "|", reason)

Wyjaśnij mi, czym jest LLM. => PROMPT | Wystarczy zwykła odpowiedź modelu.
Mam dokument PDF i chcę pytać o jego treść. => RAG | Pytanie dotyczy dokumentów lub bazy wiedzy.
Policz średnią ocen z pliku CSV. => TOOL | Pytanie wymaga obliczeń albo pracy na danych.
Chcę dostroić model do mojego stylu pisania. => TRAINING | Pytanie dotyczy uczenia lub dostrajania modelu.


# Część F. Mała sieć neuronowa do klasyfikacji workflow

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier

texts = [
    "Wyjaśnij czym jest sztuczna inteligencja", "Napisz krótką definicję LLM",
    "Co to jest transformer", "Wyjaśnij embedding prostym językiem",
    "Mam dokument PDF i chcę zadawać pytania", "Odpowiedz na podstawie regulaminu",
    "Mam bazę wiedzy i chcę z niej korzystać", "Chcę pytać o treść instrukcji BHP",
    "Policz średnią ocen studentów", "Wykonaj analizę danych z pliku CSV",
    "Oblicz odchylenie standardowe", "Zrób wykres sprzedaży miesięcznej",
    "Chcę, żeby model odpowiadał zawsze w stylu prawniczym",
    "Chcę dostroić model do mojego tonu wypowiedzi",
    "Chcę nauczyć model klasyfikować moje teksty", "Potrzebuję trenować model na przykładach"
]
labels = [
    "PROMPT", "PROMPT", "PROMPT", "PROMPT",
    "RAG", "RAG", "RAG", "RAG",
    "TOOL", "TOOL", "TOOL", "TOOL",
    "TRAINING", "TRAINING", "TRAINING", "TRAINING"
]

classifier = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("mlp", MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=1000, random_state=42))
])
classifier.fit(texts, labels)

questions = [
    "Mam dokumenty uczelni i chcę zadawać pytania o zasady zaliczenia",
    "Policz sumę sprzedaży z tabeli",
    "Wyjaśnij, czym jest RAG",
    "Chcę nauczyć model rozpoznawania mojego stylu"
]

for question in questions:
    print(question, "=>", classifier.predict([question])[0])

Mam dokumenty uczelni i chcę zadawać pytania o zasady zaliczenia => RAG
Policz sumę sprzedaży z tabeli => TOOL
Wyjaśnij, czym jest RAG => PROMPT
Chcę nauczyć model rozpoznawania mojego stylu => TRAINING


# Podsumowanie

Największy błąd początkujących: próbować wszystko trenować. W większości przypadków zaczynamy od workflow: prompt, RAG, narzędzia, walidacja, a dopiero potem ewentualny fine-tuning.